In [ ]:
import xml.etree.ElementTree as ET
import json
from typing import Dict, List

def parse_cfr_xml(xml_file: str) -> Dict:
    """
    Parse the CFR XML file and extract structured regulatory information.

    Args:
        xml_file: Path to the XML file

    Returns:
        Dictionary containing structured regulatory information
    """
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Register namespaces if present (though CFR XML typically doesn't use them)
    namespaces = {'ns': 'http://www.w3.org/2001/XMLSchema-instance'} if 'xmlns' in root.attrib else {}

    # Initialize the result structure
    result = {
        "metadata": {},
        "titles": [],
        "chapters": [],
        "parts": [],
        "sections": []
    }

    # Extract metadata
    metadata = root.find(".//TITLEPG")
    if metadata is not None:
        result["metadata"] = {
            "title_number": get_text(metadata, "TITLENUM"),
            "subject": get_text(metadata, "SUBJECT"),
            "parts": get_text(metadata, "PARTS"),
            "revised": get_text(metadata, "REVISED"),
            "contains": get_text(metadata, "CONTAINS"),
            "date": get_text(metadata, "DATE"),
            "publication": get_text(metadata, "PUB/P")
        }

    # Extract chapters
    for chapter in root.findall(".//CHAPTI"):
        chapter_data = {
            "number": get_text(chapter, "PT"),
            "subject": get_text(chapter, "SUBJECT"),
            "page": get_text(chapter, "PG")
        }
        result["chapters"].append(chapter_data)

    # Extract parts and sections
    for part in root.findall(".//PART"):
        part_data = {
            "number": clean_part_number(get_text(part, "EAR")),
            "title": get_text(part, ".//HD[@SOURCE='HED']"),
            "authority": clean_text(get_text(part, ".//AUTH/P")),
            "source": clean_text(get_text(part, ".//SOURCE/P")),
            "subparts": [],
            "sections": []
        }

        # Extract subparts if they exist
        for subpart in part.findall(".//SUBPART"):
            subpart_data = {
                "title": get_text(subpart, ".//HD[@SOURCE='HED']"),
                "sections": []
            }

            # Extract sections within subparts
            for section in subpart.findall(".//SECTION"):
                section_data = parse_section(section)
                subpart_data["sections"].append(section_data)
                result["sections"].append(section_data)

            part_data["subparts"].append(subpart_data)

        # Extract sections not in subparts (using alternative approach)
        for section in part.findall(".//SECTION"):
            # Only include if not already captured in a subpart
            if not any(section in subpart for subpart in part.findall(".//SUBPART")):
                section_data = parse_section(section)
                part_data["sections"].append(section_data)
                result["sections"].append(section_data)

        result["parts"].append(part_data)

    return result

def get_text(element: ET.Element, path: str) -> str:
    """Safe text extraction helper"""
    found = element.find(path)
    return clean_text(found.text if found is not None else "")

def clean_part_number(text: str) -> str:
    """Clean part number text"""
    return text.replace("Pt.", "").strip()

def clean_text(text: str) -> str:
    """Clean and normalize text"""
    if text is None:
        return ""
    return " ".join(text.replace("\n", " ").replace("\t", " ").strip().split())

def parse_section(section: ET.Element) -> Dict:
    """Parse an individual section element with improved paragraph handling"""
    section_data = {
        "number": get_text(section, "SECTNO"),
        "subject": get_text(section, "SUBJECT"),
        "content": [],
        "citations": []
    }

    # Extract paragraphs and subparagraphs
    current_paragraph = None
    for elem in section:
        if elem.tag == "P":
            # Handle paragraph text and any child elements
            paragraph_text = ""
            for content in elem.itertext():
                paragraph_text += content
            paragraph_text = clean_text(paragraph_text)

            if paragraph_text:
                # Check if this is a new main paragraph (starts with (a), (b), etc.)
                if (paragraph_text.strip().startswith('(') and
                    len(paragraph_text) > 1 and
                    paragraph_text[1].isalpha() and
                    len(paragraph_text) > 3 and
                    paragraph_text[2] in (')', ' ')):

                    # If we have a current paragraph, add it before starting new one
                    if current_paragraph:
                        section_data["content"].append(current_paragraph)

                    # Extract the heading (e.g., "(a)")
                    heading_end = paragraph_text.find(')') + 1
                    heading = paragraph_text[:heading_end].strip()
                    remaining_text = paragraph_text[heading_end:].strip()

                    current_paragraph = {
                        "type": "paragraph",
                        "heading": heading,
                        "text": remaining_text,
                        "subparagraphs": []
                    }
                else:
                    # Check if this is a subparagraph (i), (ii), etc.
                    if (current_paragraph and
                        paragraph_text.strip().startswith('(') and
                        any(paragraph_text[1:].startswith(num)
                            for num in ['i)', 'ii)', 'iii)', 'iv)', 'v)', 'vi)', 'vii)', 'viii)', 'ix)', 'x)'])):

                        current_paragraph["subparagraphs"].append({
                            "type": "subparagraph",
                            "text": paragraph_text
                        })
                    else:
                        # Regular paragraph content
                        if current_paragraph:
                            current_paragraph["text"] += " " + paragraph_text
                        else:
                            section_data["content"].append({
                                "type": "paragraph",
                                "text": paragraph_text
                            })
        elif elem.tag == "IPAR":
            for sub_elem in elem:
                if sub_elem.tag == "P":
                    text = clean_text(sub_elem.text or "")
                    if text:
                        section_data["content"].append({
                            "type": "indented_paragraph",
                            "text": text
                        })

    # Add the last current paragraph if it exists
    if current_paragraph:
        section_data["content"].append(current_paragraph)

    # Extract citations
    for citation in section.findall(".//CITA"):
        citation_text = clean_text(citation.text or "")
        if citation_text:
            section_data["citations"].append(citation_text)

    return section_data

def save_to_markdown(data: Dict, output_file: str) -> None:
    """Save parsed data to Markdown format with improved formatting"""
    with open(output_file, "w", encoding="utf-8") as f:
        # Write metadata
        f.write(f"# Title {data['metadata']['title_number']}: {data['metadata']['subject']}\n\n")
        f.write(f"*Revised: {data['metadata']['revised']}*\n")
        f.write(f"*Date: {data['metadata']['date']}*\n")
        f.write(f"*Parts: {data['metadata']['parts']}*\n\n")

        # Write chapters
        if data["chapters"]:
            f.write("## Chapters\n")
            for chapter in data["chapters"]:
                f.write(f"- {chapter['number']}: {chapter['subject']} (Page {chapter['page']})\n")
            f.write("\n")

        # Write parts and sections
        for part in data["parts"]:
            f.write(f"## Part {part['number']}: {part['title']}\n\n")
            if part["authority"]:
                f.write(f"**Authority:** {part['authority']}\n\n")
            if part["source"]:
                f.write(f"**Source:** {part['source']}\n\n")

            # Write sections not in subparts
            for section in part["sections"]:
                write_section_markdown(f, section)

            # Write subparts and their sections
            for subpart in part["subparts"]:
                f.write(f"### Subpart: {subpart['title']}\n\n")
                for section in subpart["sections"]:
                    write_section_markdown(f, section)

def write_section_markdown(file, section: Dict):
    """Helper to write a section in Markdown format with improved structure"""
    file.write(f"#### § {section['number']} {section['subject']}\n\n")

    for content in section["content"]:
        if content["type"] == "paragraph":
            if "heading" in content:
                # Main paragraph with heading (a), (b), etc.
                file.write(f"**{content['heading']}** {content['text']}\n\n")

                # Write subparagraphs if they exist
                for subpara in content.get("subparagraphs", []):
                    # Indent subparagraphs and make them italic
                    file.write(f"    *{subpara['text']}*\n\n")
            else:
                # Regular paragraph without heading
                file.write(f"{content['text']}\n\n")

        elif content["type"] == "indented_paragraph":
            # Indented paragraphs (from IPAR tags)
            file.write(f"    {content['text']}\n\n")

    # Write citations if they exist
    if section["citations"]:
        file.write("**Citations:**\n")
        for citation in section["citations"]:
            file.write(f"- {citation}\n")
        file.write("\n")

def main():
    input_file = "/content/drive/MyDrive/CIS 630 - Final Project/Source Data - CFR File/CFR-2024/title-12/CFR-2024-title12-vol1.xml"
    md_output = "/content/drive/MyDrive/CIS 630 - Final Project/Pram XML Parser/CFR_parsed_output/title12-vol1_parsed_cfr.md"

    print(f"Parsing {input_file}...")
    parsed_data = parse_cfr_xml(input_file)

    print(f"Saving Markdown to {md_output}...")
    print(parsed_data)
    save_to_markdown(parsed_data, md_output)

    print("Done!")

if __name__ == "__main__":
    main()

Parsing title-12/CFR-2024-title12-vol1.xml...
Saving Markdown to cfr_parsed_output/title12-vol1_parsed_cfr.md...
dict_keys(['metadata', 'titles', 'chapters', 'parts', 'sections'])
Done!
